In [16]:
!pip install pyspark -q

In [17]:
import pandas as pd
import numpy as np
import os
import shutil
import time

from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [19]:
df = pd.read_excel("/content/patients_data_with_alerts.xlsx")

df.head()

,Patient Number,Heart Rate (bpm),SpO2 Level (%),Systolic Blood Pressure (mmHg),Diastolic Blood Pressure (mmHg),Body Temperature (°C),Fall Detection,Predicted Disease,Data Accuracy (%),Heart Rate Alert,SpO2 Level Alert,Blood Pressure Alert,Temperature Alert
0,1,96,92,101,89,36.904680,No,Diabetes Mellitus,95,Normal,Normal,Normal,Normal
1,2,76,83,103,85,37.254129,Yes,Asthma,91,Normal,Low,Normal,Abnormal
2,3,92,88,123,70,36.539418,Yes,Asthma,86,Normal,Low,Normal,Normal
3,4,137,84,167,97,37.018767,Yes,Asthma,99,High,Low,High,Normal
4,5,76,81,175,80,37.328671,No,Normal,93,Normal,Low,High,Abnormal


In [20]:
patients = df[['Patient Number']].head(20)

rows = []

start = datetime(2026,1,1,0,0,0)

for t in range(300):

    for p in patients['Patient Number']:

        if p <= 5:
            hr = np.random.randint(105,120)
        else:
            hr = np.random.randint(70,90)

        rows.append([
            p,
            hr,
            start+timedelta(seconds=t)
        ])

stream_df = pd.DataFrame(
    rows,
    columns=[
        "patient_id",
        "heart_rate",
        "event_time"
    ]
)

stream_df.head()

,patient_id,heart_rate,event_time
0,1,109,2026-01-01
1,2,109,2026-01-01
2,3,109,2026-01-01
3,4,108,2026-01-01
4,5,111,2026-01-01


In [21]:
base="/content/hospital_stream"

input_path=base+"/input"

batch_path=base+"/batch"

checkpoint=base+"/checkpoint"

shutil.rmtree(base,ignore_errors=True)

os.makedirs(input_path)

os.makedirs(batch_path)

os.makedirs(checkpoint)

In [22]:
batch=500

for i in range(0,len(stream_df),batch):

    stream_df.iloc[i:i+batch].to_csv(
        f"{batch_path}/batch_{i}.csv",
        index=False
    )

len(os.listdir(batch_path))

12

In [45]:
spark=SparkSession.builder\
.appName("Hospital")\
.master("local[*]")\
.getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [44]:
schema=StructType([

StructField("patient_id",IntegerType(),True),

StructField("heart_rate",IntegerType(),True),

StructField("event_time",TimestampType(),True)

])

In [43]:
stream=spark.readStream\
.option("header",True)\
.schema(schema)\
.csv(input_path)

In [42]:
windowed=stream\
.withWatermark(
"event_time",
"2 minutes"
)\
.groupBy(

col("patient_id"),

window(
col("event_time"),
"2 minutes"
)

)\
.agg(

avg("heart_rate").alias("avg_hr")

)

In [36]:
alerts=windowed.filter(

col("avg_hr")>100

)

In [51]:
#ss
query = alerts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", checkpoint + "_new") \
    .start()

In [29]:
files=sorted(os.listdir(batch_path))

for f in files:

    shutil.copy(

        os.path.join(batch_path,f),

        os.path.join(input_path,f)

    )

    print(f)

    time.sleep(2)

batch_0.csv
batch_1000.csv
batch_1500.csv
batch_2000.csv
batch_2500.csv
batch_3000.csv
batch_3500.csv
batch_4000.csv
batch_4500.csv
batch_500.csv
batch_5000.csv
batch_5500.csv


In [30]:
time.sleep(20)

query.stop()

In [39]:
query.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [40]:
spark.streams.active

In [41]:
query.processAllAvailable()

In [50]:
try:
    query.stop()
except:
    pass

# clear input and checkpoint
shutil.rmtree(input_path, ignore_errors=True)
shutil.rmtree(checkpoint + "_final", ignore_errors=True)

os.makedirs(input_path, exist_ok=True)

# start stream again
query = alerts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", checkpoint + "_final") \
    .start()

# copy files AFTER stream starts
files = sorted(os.listdir(batch_path))

for f in files:
    shutil.copy(
        os.path.join(batch_path, f),
        os.path.join(input_path, f)
    )
    print("Added:", f)
    time.sleep(2)

query.processAllAvailable()

Added: batch_0.csv
Added: batch_1000.csv
Added: batch_1500.csv
Added: batch_2000.csv
Added: batch_2500.csv
Added: batch_3000.csv
Added: batch_3500.csv
Added: batch_4000.csv
Added: batch_4500.csv
Added: batch_500.csv
Added: batch_5000.csv
Added: batch_5500.csv


In [52]:
try:
    query.stop()
except:
    pass

shutil.rmtree(input_path, ignore_errors=True)
shutil.rmtree(checkpoint + "_print", ignore_errors=True)

os.makedirs(input_path, exist_ok=True)

def show_alerts(batch_df, batch_id):
    print("====================================")
    print("ALERT BATCH:", batch_id)
    print("====================================")
    batch_df.show(truncate=False)

query = alerts.writeStream \
    .outputMode("complete") \
    .foreachBatch(show_alerts) \
    .option("checkpointLocation", checkpoint + "_print") \
    .start()

files = sorted(os.listdir(batch_path))

for f in files:
    shutil.copy(
        os.path.join(batch_path, f),
        os.path.join(input_path, f)
    )
    print("Added:", f)
    time.sleep(2)

query.processAllAvailable()

Added: batch_0.csv
ALERT BATCH: 0
Added: batch_1000.csv
Added: batch_1500.csv
Added: batch_2000.csv
Added: batch_2500.csv
Added: batch_3000.csv
Added: batch_3500.csv
+----------+------------------------------------------+------+
|patient_id|window                                    |avg_hr|
+----------+------------------------------------------+------+
|5         |{2026-01-01 00:00:00, 2026-01-01 00:02:00}|113.48|
|3         |{2026-01-01 00:00:00, 2026-01-01 00:02:00}|111.32|
|1         |{2026-01-01 00:00:00, 2026-01-01 00:02:00}|112.16|
|2         |{2026-01-01 00:00:00, 2026-01-01 00:02:00}|111.68|
|4         |{2026-01-01 00:00:00, 2026-01-01 00:02:00}|113.56|
+----------+------------------------------------------+------+

ALERT BATCH: 1
Added: batch_4000.csv
Added: batch_4500.csv
Added: batch_500.csv
Added: batch_5000.csv
Added: batch_5500.csv
+----------+------------------------------------------+------------------+
|patient_id|window                                    |avg_hr      